# M43 lab solution: a domain knowledge graph

**Brief.** Domain KG: extract a graph from 20 documents in your domain, load it into Neo4j, and answer five multi-hop questions that vector RAG gets wrong.

The domain is Larkspur Cloud: its 13 handbook files plus seven internal records in `../data/extra_docs/` (three teams, four incidents), 20 documents in all. The steps:

1. Start Neo4j with [`docker-compose.yml`](docker-compose.yml).
2. Extract a graph from the 20 documents with `LLMGraphTransformer` (as notebook `05b_create_kg_from_text.ipynb` does), restricted to a small schema, and load it with `Neo4jGraph.add_graph_documents`.
3. Answer the five questions in `../data/multihop.jsonl` two ways: through the graph (text-to-Cypher with `GraphCypherQAChain`, as in lesson 43.3, plus the two most relevant handbook passages for policy text the graph does not hold), and with plain vector RAG over the same 20 documents.
4. Grade both with an LLM judge against the reference answers, and show for each vector failure which documents on the answer's path it never retrieved.

## Setup

Start Neo4j first, from this folder: `docker compose up -d`. The defaults below match the compose file; set `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD` to use another instance.

In [ ]:
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_embeddings, get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl  # noqa: E402

DATA = Path("../data")
llm = get_experientiallabs_llm()
embeddings = get_embeddings()
MODEL = llm.model_name

pages = load_handbook(DATA / "larkspur" / "handbook") + load_handbook(DATA / "extra_docs")
docs = [Document(page_content=f"{p['title']}\n\n{p['text']}", metadata={"source": p["source"]}) for p in pages]
questions = load_jsonl(DATA / "multihop.jsonl")
assert len(docs) == 20, len(docs)

graph = Neo4jGraph(
    url=os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
    username=os.environ.get("NEO4J_USERNAME", "neo4j"),
    password=os.environ.get("NEO4J_PASSWORD", "larkspur-lab"),  # the local container's password from docker-compose.yml
)
print("model:", MODEL, "|", len(docs), "documents |", len(questions), "questions")

## Extract the graph

The schema is small on purpose: the node and relationship types the questions need. `strict_mode` drops anything outside it, and incidents keep their date and monthly availability as properties.

In [ ]:
NODES = ["Team", "Person", "Service", "Region", "Incident"]
RELS = [
    ("Person", "LEADS", "Team"),
    ("Team", "OWNS", "Service"),
    ("Team", "ON_CALL_FOR", "Region"),
    ("Incident", "AFFECTED", "Service"),
    ("Incident", "IN_REGION", "Region"),
]
transformer = LLMGraphTransformer(llm=llm, allowed_nodes=NODES, allowed_relationships=RELS,
                                  node_properties=["date", "availability"], strict_mode=True)

start = time.perf_counter()
graph_docs = transformer.convert_to_graph_documents(docs)
extract_s = time.perf_counter() - start
print(f"extracted in {extract_s:.1f}s:",
      sum(len(g.nodes) for g in graph_docs), "nodes,", sum(len(g.relationships) for g in graph_docs), "relationships")
for g in graph_docs:
    if g.relationships:
        print(g.source.metadata["source"], [(r.source.id, r.type, r.target.id) for r in g.relationships])

In [ ]:
graph.query("MATCH (n) DETACH DELETE n")  # start from an empty database on every run
graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=True)
graph.refresh_schema()
print(graph.query("MATCH (n) WHERE NOT n:Document RETURN labels(n) AS labels, count(*) AS n ORDER BY n DESC"))
print(graph.schema)

## Check the paths with hand-written Cypher

Before any model writes Cypher, these queries show whether the extracted graph holds each question's path. Node ids are whatever the extractor chose, so they match on lower-cased substrings.

In [ ]:
PATHS = {
    "m1": """MATCH (i:Incident)-[:AFFECTED]->(s:Service)<-[:OWNS]-(t:Team)<-[:LEADS]-(p:Person)
             WHERE toLower(i.id) CONTAINS '104' RETURN i.id, s.id, t.id, p.id""",
    "m2": """MATCH (t:Team)-[:ON_CALL_FOR]->(r:Region)<-[:IN_REGION]-(i:Incident)
             WHERE toLower(t.id) CONTAINS 'network' RETURN t.id, r.id, i.id, i.date""",
    "m3": """MATCH (i:Incident)-[:IN_REGION]->(r:Region) WHERE toLower(i.id) CONTAINS '131'
             RETURN i.id, r.id, i.availability, i.date""",
    "m4": """MATCH (i:Incident)-[:AFFECTED]->(s:Service)<-[:OWNS]-(t:Team)<-[:LEADS]-(p:Person)
             WHERE toLower(i.id) CONTAINS '140' RETURN i.id, s.id, t.id, p.id""",
    "m5": """MATCH (p:Person)-[:LEADS]->(t:Team) WHERE toLower(p.id) CONTAINS 'priya'
             MATCH (i:Incident) WHERE (i)-[:AFFECTED]->(:Service)<-[:OWNS]-(t) OR (i)-[:IN_REGION]->(:Region)<-[:ON_CALL_FOR]-(t)
             RETURN DISTINCT i.id, i.availability""",
}
for qid, cypher in PATHS.items():
    print(qid, graph.query(cypher))

## Route 1: graph queries (text-to-Cypher)

`GraphCypherQAChain` asks the model for a Cypher query from the graph schema, runs it, and returns the rows. The rows and the two handbook passages closest to the question (for policy text such as credit rates and refund times, which the schema does not model) go to one answer call.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
store = InMemoryVectorStore.from_documents(chunks, embeddings)
handbook_store = InMemoryVectorStore.from_documents(
    [c for c in chunks if c.metadata["source"][:2] < "14"], embeddings)  # handbook files only, 01 to 13

cypher_chain = GraphCypherQAChain.from_llm(llm, graph=graph, validate_cypher=True, allow_dangerous_requests=True,
                                           return_intermediate_steps=True, top_k=20)


def answer_from(context: str, question: str) -> str:
    return llm.invoke([("system", "Answer from the context only. If it does not answer, say what is missing.\n\n"
                                  "Context:\n" + context), ("human", question)]).content


def graph_route(question: str) -> dict:
    try:
        out = cypher_chain.invoke({"query": question})
        steps = out["intermediate_steps"]
        cypher, rows = steps[0]["query"], steps[1]["context"]
    except Exception as exc:  # noqa: BLE001  a Cypher error is a graph-route failure, not a crash
        cypher, rows = f"(failed: {exc})", []
    policy = handbook_store.similarity_search(question, k=2)
    context = f"Graph rows for the query {cypher}:\n{rows}\n\nHandbook passages:\n" + "\n---\n".join(
        d.page_content for d in policy)
    return {"answer": answer_from(context, question), "cypher": cypher, "rows": rows,
            "sources": sorted({d.metadata["source"] for d in policy})}

## Route 2: vector RAG over the same 20 documents

In [ ]:
def vector_route(question: str, k: int = 4) -> dict:
    hits = store.similarity_search(question, k=k)
    return {"answer": answer_from("\n---\n".join(d.page_content for d in hits), question),
            "sources": sorted({d.metadata["source"] for d in hits})}

## Judge and run

In [ ]:
JUDGE_PROMPT = """You grade answers to multi-hop questions about a company's internal records and handbook.

Question: {question}
Reference answer (complete and correct): {reference}
Answer to grade: {answer}

Score 1 to 5: 5 = every fact of the reference and no contradiction; 4 = the key fact right, a detail missing;
3 = partly right; 2 = mostly wrong or a wrong key fact; 1 = wrong or no answer. Give one sentence of reason."""


class Verdict(BaseModel):
    score: int = Field(ge=1, le=5)
    reason: str


judge = llm.with_structured_output(Verdict)
print(JUDGE_PROMPT)

results = []
for q in questions:
    for route, fn in (("graph", graph_route), ("vector", vector_route)):
        r = fn(q["question"])
        v = judge.invoke(JUDGE_PROMPT.format(question=q["question"], reference=q["reference_answer"], answer=r["answer"]))
        results.append({"id": q["id"], "route": route, "score": v.score, "reason": v.reason, **r})
    print(q["id"], [(x["route"], x["score"]) for x in results[-2:]])

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df["correct"] = df["score"] >= 4
print("model:", MODEL, "| embeddings:", type(embeddings).__name__, "| run:", time.strftime("%Y-%m-%d"))
print(f"graph extraction: {extract_s:.1f}s for {len(docs)} documents")
print(df.pivot(index="id", columns="route", values="score").to_markdown())
print(df.groupby("route")["correct"].sum().to_markdown())

## The vector failures

For every question vector RAG got wrong (score below 4): the documents on the answer's path (from `hops`), the ones its top-k retrieval never returned, and the judge's reason. The graph route's Cypher is printed beside it.

In [ ]:
by_id = {q["id"]: q for q in questions}
for r in df[(df["route"] == "vector") & (~df["correct"])].itertuples():
    path = sorted({h.split(":")[0] for h in by_id[r.id]["hops"]})
    missing = [p for p in path if p not in r.sources]
    g = df[(df["id"] == r.id) & (df["route"] == "graph")].iloc[0]
    print(f"{r.id}: {by_id[r.id]['question']}")
    print(f"  path documents: {path}\n  vector retrieved: {r.sources}\n  never retrieved: {missing}")
    print(f"  vector answer ({r.score}): {r.answer}\n  judge: {r.reason}")
    print(f"  graph Cypher: {g['cypher']}\n  graph answer ({g['score']}): {g['answer']}\n")

## What this shows

- Each question's answer sits on a path across two to four documents. Vector RAG answers it only when every document on the path lands in its top k; the failure printout names the documents it missed.
- The graph route follows the path in one Cypher query, but only as far as extraction built it. If a hand-written query above returns no rows, the gap is in extraction (a missing node or relationship), not in Cypher generation.
- Policy text such as credit rates is not in the schema, which is why the graph route also reads two handbook passages. A schema that modelled policies would move that join into the graph too.